In [1]:
# Cell 1 — Install Dependencies
!pip install openai langchain langchain-openai chromadb sentence-transformers newspaper3k newsapi-python python-dotenv -q

print("✅ Dependencies installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 64.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 73.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14

In [3]:
# Cell 2 — Setup API Key Safely
from google.colab import userdata
import os

# Store your API key in Colab Secrets (safer than hardcoding)
# Click the 🔑 key icon on the left sidebar in Colab
# Add secret name: OPENAI_API_KEY
# Add your API key as the value

try:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("✅ API key loaded from Colab Secrets!")
except:
    # Fallback — paste directly (only for testing)
    os.environ["OPENAI_API_KEY"] = "paste-your-key-here"
    print("⚠️ API key set directly — move to Colab Secrets later!")

✅ API key loaded from Colab Secrets!


In [11]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
response = llm.invoke([HumanMessage(content="Say 'InfoTrace RAG system is ready!' in one line")])
print(response.content)

InfoTrace RAG system is ready!


In [12]:
!pip install lxml_html_clean -q
print("✅ Fixed!")

✅ Fixed!


In [13]:
import chromadb
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
chroma_client = chromadb.Client()

# Get or create — won't fail if already exists
collection = chroma_client.get_or_create_collection(name="infotrace_knowledge")

print("✅ ChromaDB collection ready!")
print(f"📚 Embedding model: all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ ChromaDB collection ready!
📚 Embedding model: all-MiniLM-L6-v2


In [14]:
from newspaper import Article
import time

def fetch_verified_facts(topic):
    sources = [
        f"https://www.bbc.com/search?q={topic.replace(' ', '+')}",
        f"https://www.reuters.com/search/news?blob={topic.replace(' ', '+')}",
    ]

    articles = []

    # Sample verified articles for testing
    # (Real scraping will be added in production)
    sample_facts = [
        {
            "source": "BBC News",
            "title": f"Latest updates on {topic}",
            "content": f"According to verified reports, {topic} has been covered extensively by international media with confirmed facts and figures from official sources.",
            "url": "https://bbc.com"
        },
        {
            "source": "Reuters",
            "title": f"Reuters coverage of {topic}",
            "content": f"Reuters correspondents have verified key facts about {topic} through official government and institutional sources.",
            "url": "https://reuters.com"
        }
    ]

    print(f"✅ Fetched {len(sample_facts)} verified sources for: '{topic}'")
    return sample_facts

# Test it
topic = "flood in Assam"
facts = fetch_verified_facts(topic)
for f in facts:
    print(f"\n📰 Source: {f['source']}")
    print(f"   Title: {f['title']}")

✅ Fetched 2 verified sources for: 'flood in Assam'

📰 Source: BBC News
   Title: Latest updates on flood in Assam

📰 Source: Reuters
   Title: Reuters coverage of flood in Assam


In [15]:
def store_in_vectordb(articles, topic, source_type):
    for i, article in enumerate(articles):
        embedding = embedding_model.encode(article["content"]).tolist()

        collection.add(
            embeddings=[embedding],
            documents=[article["content"]],
            metadatas=[{
                "source": article["source"],
                "title": article["title"],
                "url": article["url"],
                "topic": topic,
                "source_type": source_type
            }],
            ids=[f"{source_type}_{topic.replace(' ', '_')}_{i}"]
        )

    print(f"✅ Stored {len(articles)} {source_type} documents in ChromaDB!")

# Test it
store_in_vectordb(facts, topic, "verified_news")

# Check whats in the database
print(f"\n📊 Total documents in ChromaDB: {collection.count()}")

✅ Stored 2 verified_news documents in ChromaDB!

📊 Total documents in ChromaDB: 2


In [16]:
def simulate_platform_posts(topic):
    platform_posts = {
        "facebook": [
            {
                "content": f"BREAKING: Government is hiding the real truth about {topic}! Share before they delete this!!",
                "likes": 1500,
                "shares": 890
            },
            {
                "content": f"My cousin who lives there says the situation with {topic} is much worse than what media is showing us.",
                "likes": 432,
                "shares": 211
            },
            {
                "content": f"Official reports confirm {topic} has affected thousands. Relief efforts are underway by local authorities.",
                "likes": 234,
                "shares": 89
            }
        ],
        "instagram": [
            {
                "content": f"Pray for everyone affected 🙏 {topic} is heartbreaking. Share awareness! #pray #help",
                "likes": 4500,
                "shares": 1200
            },
            {
                "content": f"Real footage of {topic} that mainstream media won't show you. Stay informed! #truth #wakeup",
                "likes": 2300,
                "shares": 670
            },
            {
                "content": f"Relief organizations are on ground helping victims of {topic}. Here is how you can help.",
                "likes": 1800,
                "shares": 430
            }
        ]
    }

    print(f"✅ Simulated posts for: '{topic}'")
    for platform, posts in platform_posts.items():
        print(f"   {platform}: {len(posts)} posts")

    return platform_posts

# Test it
platform_posts = simulate_platform_posts(topic)

✅ Simulated posts for: 'flood in Assam'
   facebook: 3 posts
   instagram: 3 posts


In [18]:
def analyze_platform_posts(platform_posts, collection):
    results = {}

    for platform, posts in platform_posts.items():
        platform_scores = []

        for post in posts:
            # Convert post to embedding
            post_embedding = embedding_model.encode(post["content"]).tolist()

            # Query ChromaDB for similar verified facts
            query_results = collection.query(
                query_embeddings=[post_embedding],
                n_results=1
            )

            # Get similarity score (1 - distance)
            distance = query_results["distances"][0][0]
            similarity = round(1 - distance, 4)

            platform_scores.append({
                "post": post["content"],
                "similarity_to_facts": similarity,
                "likes": post["likes"],
                "shares": post["shares"]
            })

        # Average similarity score for platform
        avg_similarity = round(
            sum(p["similarity_to_facts"] for p in platform_scores) / len(platform_scores), 4
        )

        results[platform] = {
            "posts": platform_scores,
            "average_similarity": avg_similarity,
            "total_posts": len(posts)
        }

        print(f"\n📊 {platform.upper()} Analysis:")
        print(f"   Average similarity to verified facts: {avg_similarity}")
        for i, p in enumerate(platform_scores):
            print(f"   Post {i+1}: {p['similarity_to_facts']} similarity | {p['likes']} likes")

    return results

# Test it
analysis_results = analyze_platform_posts(platform_posts, collection)



📊 FACEBOOK Analysis:
   Average similarity to verified facts: 0.4451
   Post 1: 0.3944 similarity | 1500 likes
   Post 2: 0.4644 similarity | 432 likes
   Post 3: 0.4766 similarity | 234 likes

📊 INSTAGRAM Analysis:
   Average similarity to verified facts: 0.196
   Post 1: 0.0699 similarity | 4500 likes
   Post 2: 0.4791 similarity | 2300 likes
   Post 3: 0.0391 similarity | 1800 likes


In [19]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

def generate_infotrace_report(topic, analysis_results):

    summary = ""
    for platform, data in analysis_results.items():
        summary += f"\n{platform.upper()}:\n"
        summary += f"  Average similarity to verified facts: {data['average_similarity']}\n"
        for i, post in enumerate(data["posts"]):
            summary += f"  Post {i+1}: similarity={post['similarity_to_facts']} | likes={post['likes']} | shares={post['shares']}\n"
            summary += f"  Content: {post['post'][:100]}...\n"

    prompt = f"""
You are InfoTrace, an intelligent information analysis system.

Topic being analyzed: "{topic}"

Here is how this topic is being discussed across social media platforms,
with similarity scores showing how closely each post aligns with verified facts
(1.0 = perfectly aligned, 0.0 = completely diverged from facts):

{summary}

Please provide:
1. A clear comparison of how each platform is discussing this topic
2. Which platform is most aligned with facts and which is most distorted
3. What kind of misinformation patterns you notice
4. Which posts are most dangerous (high distortion + high engagement)
5. A brief InfoTrace score summary for each platform

Be specific, insightful and use the data provided.
"""

    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content

# Generate report
print("🔍 Generating InfoTrace Report...\n")
report = generate_infotrace_report(topic, analysis_results)
print(report)

🔍 Generating InfoTrace Report...

1. Facebook posts generally have a higher average similarity to verified facts compared to Instagram posts. Facebook posts tend to provide more information about the flood in Assam, including official reports and relief efforts. On the other hand, Instagram posts focus more on emotional responses, sharing awareness, and questioning mainstream media coverage.

2. Facebook is most aligned with facts with an average similarity score of 0.4451, while Instagram is the most distorted with an average similarity score of 0.196.

3. Misinformation patterns on Facebook include claims of the government hiding the truth and exaggerating the situation, while on Instagram, there is a focus on emotional appeals, sharing unverified footage, and questioning mainstream media credibility.

4. The most dangerous posts are Post 1 on Facebook (similarity=0.3944, likes=1500, shares=890) and Post 2 on Instagram (similarity=0.4791, likes=2300, shares=670) as they have high eng

In [20]:
from transformers import pipeline

sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    return_all_scores=True
)

print("✅ Sentiment analyzer loaded!")

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

✅ Sentiment analyzer loaded!


In [22]:
def analyze_sentiment_per_platform(platform_posts):
    sentiment_results = {}

    for platform, posts in platform_posts.items():
        platform_sentiments = []

        for post in posts:
            # Get sentiment scores
            raw = sentiment_analyzer(post["content"][:512])

            # Handle both list formats
            if isinstance(raw[0], list):
                scores = raw[0]
            else:
                scores = raw

            # Extract scores
            sentiment_dict = {s["label"]: round(s["score"], 4) for s in scores}

            platform_sentiments.append({
                "post": post["content"][:80] + "...",
                "sentiment": sentiment_dict,
                "dominant": max(sentiment_dict, key=sentiment_dict.get),
                "likes": post["likes"],
                "shares": post["shares"]
            })

        # Calculate platform level averages
        avg_positive = round(sum(p["sentiment"].get("positive", 0) for p in platform_sentiments) / len(platform_sentiments), 4)
        avg_negative = round(sum(p["sentiment"].get("negative", 0) for p in platform_sentiments) / len(platform_sentiments), 4)
        avg_neutral  = round(sum(p["sentiment"].get("neutral", 0)  for p in platform_sentiments) / len(platform_sentiments), 4)

        sentiment_results[platform] = {
            "posts": platform_sentiments,
            "average": {
                "positive": avg_positive,
                "negative": avg_negative,
                "neutral":  avg_neutral
            }
        }

        print(f"\n💬 {platform.upper()} Sentiment:")
        print(f"   Positive: {avg_positive} | Negative: {avg_negative} | Neutral: {avg_neutral}")
        for i, p in enumerate(platform_sentiments):
            print(f"   Post {i+1}: {p['dominant']} | likes={p['likes']}")

    return sentiment_results

# Test it
sentiment_results = analyze_sentiment_per_platform(platform_posts)


💬 FACEBOOK Sentiment:
   Positive: 0.0 | Negative: 0.7237 | Neutral: 0.0
   Post 1: negative | likes=1500
   Post 2: negative | likes=432
   Post 3: negative | likes=234

💬 INSTAGRAM Sentiment:
   Positive: 0.0 | Negative: 0.1775 | Neutral: 0.4838
   Post 1: negative | likes=4500
   Post 2: neutral | likes=2300
   Post 3: neutral | likes=1800


In [24]:
!pip install bertopic -q
print("✅ BERTopic installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 10.9 MB/s eta 0:00:00
✅ BERTopic installed!


In [25]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

def analyze_topics_per_platform(platform_posts):
    topic_results = {}

    for platform, posts in platform_posts.items():
        # Extract post contents
        docs = [post["content"] for post in posts]

        # Need minimum 3 docs for BERTopic
        if len(docs) < 3:
            print(f"⚠️ {platform}: Not enough posts for topic modeling")
            continue

        # Use simple keyword extraction for small datasets
        vectorizer = CountVectorizer(
            ngram_range=(1, 2),
            stop_words="english",
            max_features=10
        )

        try:
            X = vectorizer.fit_transform(docs)
            keywords = vectorizer.get_feature_names_out()

            topic_results[platform] = {
                "dominant_keywords": list(keywords),
                "total_posts": len(docs)
            }

            print(f"\n🔍 {platform.upper()} Top Keywords:")
            print(f"   {', '.join(keywords[:8])}")

        except Exception as e:
            print(f"⚠️ {platform} topic modeling failed: {e}")

    return topic_results

# Test it
topic_results = analyze_topics_per_platform(platform_posts)

/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.



🔍 FACEBOOK Top Keywords:
   affected, affected thousands, assam, assam affected, assam share, assam worse, breaking government, confirm

🔍 INSTAGRAM Top Keywords:
   affected, assam, assam help, assam mainstream, awareness, awareness pray, flood, flood assam


In [26]:

def infotrace_orchestrator(user_topic):
    print(f"🔍 InfoTrace Analysis Started")
    print(f"📌 Topic: '{user_topic}'")
    print("="*50)

    # Step 1 — Fetch verified facts
    print("\n📰 Step 1: Fetching verified facts...")
    facts = fetch_verified_facts(user_topic)
    store_in_vectordb(facts, user_topic, "verified_news")

    # Step 2 — Collect platform posts
    print("\n📱 Step 2: Collecting platform posts...")
    posts = simulate_platform_posts(user_topic)

    # Step 3 — Similarity analysis
    print("\n🔎 Step 3: Analyzing similarity to verified facts...")
    analysis = analyze_platform_posts(posts, collection)

    # Step 4 — Sentiment analysis
    print("\n💬 Step 4: Analyzing sentiment per platform...")
    sentiment = analyze_sentiment_per_platform(posts)

    # Step 5 — Topic modeling
    print("\n🧠 Step 5: Extracting dominant narratives...")
    topics = analyze_topics_per_platform(posts)

    # Step 6 — Generate final report
    print("\n📄 Step 6: Generating InfoTrace Report...")
    report = generate_infotrace_report(user_topic, analysis)

    # Compile full results
    full_results = {
        "topic": user_topic,
        "similarity": analysis,
        "sentiment": sentiment,
        "topics": topics,
        "report": report
    }

    print("\n" + "="*50)
    print("✅ InfoTrace Analysis Complete!")
    print("="*50)
    print(f"\n{report}")

    return full_results

# Test the full pipeline with one prompt!
results = infotrace_orchestrator("flood in Assam")

🔍 InfoTrace Analysis Started
📌 Topic: 'flood in Assam'

📰 Step 1: Fetching verified facts...
✅ Fetched 2 verified sources for: 'flood in Assam'
✅ Stored 2 verified_news documents in ChromaDB!

📱 Step 2: Collecting platform posts...
✅ Simulated posts for: 'flood in Assam'
   facebook: 3 posts
   instagram: 3 posts

🔎 Step 3: Analyzing similarity to verified facts...

📊 FACEBOOK Analysis:
   Average similarity to verified facts: 0.4451
   Post 1: 0.3944 similarity | 1500 likes
   Post 2: 0.4644 similarity | 432 likes
   Post 3: 0.4766 similarity | 234 likes

📊 INSTAGRAM Analysis:
   Average similarity to verified facts: 0.196
   Post 1: 0.0699 similarity | 4500 likes
   Post 2: 0.4791 similarity | 2300 likes
   Post 3: 0.0391 similarity | 1800 likes

💬 Step 4: Analyzing sentiment per platform...

💬 FACEBOOK Sentiment:
   Positive: 0.0 | Negative: 0.7237 | Neutral: 0.0
   Post 1: negative | likes=1500
   Post 2: negative | likes=432
   Post 3: negative | likes=234

💬 INSTAGRAM Sentiment:
